In [ ]:
from typing import TypedDict

class ProjectState(TypedDict):
    # SESSION
    session_id: str

    # PROJECT
    project_id: str
    prompt: str

    # SCOPE
    scope_allowed: bool
    scope_reason: str

    # PLANNING
    plan: dict
    plan_summary: str

    # HUMAN LOOP
    human_message: str
    action: str
    approved: bool

    # GENERATION
    generated: bool

    # BUILD
    build_success: bool
    build_output: str
    build_error: str

    # REVIEW / FIX
    fix_instructions: str
    retry_count: int

    # PREVIEW
    preview_url: str

    # STATUS
    status: str

    assistant_response: str

    

In [ ]:
from langgraph.graph import StateGraph, END

builder = StateGraph(ProjectState)

builder.add_node("Scope Agent", lambda s: s)
builder.add_node("Planner Agent", lambda s: s)

builder.set_entry_point("Scope Agent")

builder.add_conditional_edges(
    "Scope Agent",
    lambda s: "Planner Agent" if s["scope_allowed"] else END,
    {
        "Planner Agent": "Planner Agent",
        END: END,
    },
)

builder.add_edge("Planner Agent", END)

planning_doc_graph = builder.compile()


from IPython.display import Image, display

display(
    Image(
        planning_doc_graph.get_graph().draw_mermaid_png()
    )
)

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(ProjectState)

builder.add_node("Human Feedback", lambda s: s)

builder.add_node("Intent Agent", lambda s: s)
builder.add_node("Scope Agent", lambda s: s)
builder.add_node("Plan Editor Agent", lambda s: s)

builder.add_node("Generator Agent", lambda s: s)
builder.add_node("Build Agent", lambda s: s)
builder.add_node("Reviewer Agent", lambda s: s)
builder.add_node("Fix Agent", lambda s: s)
builder.add_node("Preview Agent", lambda s: s)

builder.add_edge(START, "Human Feedback")
builder.add_edge("Human Feedback", "Intent Agent")

builder.add_conditional_edges(
    "Intent Agent",
    lambda s: (
        "Generator Agent"
        if s["action"] == "approve"
        else "Scope Agent"
    ),
    {
        "Generator Agent": "Generator Agent",
        "Scope Agent": "Scope Agent",
    },
)

builder.add_conditional_edges(
    "Scope Agent",
    lambda s: (
        "Plan Editor Agent"
        if s["scope_allowed"]
        else END
    ),
    {
        "Plan Editor Agent": "Plan Editor Agent",
        END: END,
    },
)

builder.add_edge("Plan Editor Agent", END)

builder.add_edge("Generator Agent", "Build Agent")

builder.add_conditional_edges(
    "Build Agent",
    lambda s: (
        "Preview Agent"
        if s["build_success"]
        else (
            END
            if s["retry_count"] >= 3
            else "Reviewer Agent"
        )
    ),
    {
        "Preview Agent": "Preview Agent",
        "Reviewer Agent": "Reviewer Agent",
        END: END,
    },
)

builder.add_edge("Reviewer Agent", "Fix Agent")
builder.add_edge("Fix Agent", "Build Agent")
builder.add_edge("Preview Agent", END)

approval_execution_graph = builder.compile()

from IPython.display import Image, display

display(
    Image(
        approval_execution_graph.get_graph().draw_mermaid_png()
    )
)

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(ProjectState)

builder.add_node("Human Request", lambda s: s)

builder.add_node("Modify Intent Agent", lambda s: s)
builder.add_node("Scope Agent", lambda s: s)
builder.add_node("Editor Agent", lambda s: s)

builder.add_node("Build Agent", lambda s: s)
builder.add_node("Reviewer Agent", lambda s: s)
builder.add_node("Fix Agent", lambda s: s)
builder.add_node("Preview Agent", lambda s: s)

builder.add_node("Satisfaction Agent", lambda s: s)

builder.add_edge(START, "Human Request")
builder.add_edge("Human Request", "Modify Intent Agent")

builder.add_conditional_edges(
    "Modify Intent Agent",
    lambda s: (
        "Satisfaction Agent"
        if s["action"] == "approve"
        else "Scope Agent"
    ),
    {
        "Satisfaction Agent": "Satisfaction Agent",
        "Scope Agent": "Scope Agent",
    },
)

builder.add_edge("Satisfaction Agent", END)

builder.add_conditional_edges(
    "Scope Agent",
    lambda s: (
        "Editor Agent"
        if s["scope_allowed"]
        else END
    ),
    {
        "Editor Agent": "Editor Agent",
        END: END,
    },
)

builder.add_edge("Editor Agent", "Build Agent")

builder.add_conditional_edges(
    "Build Agent",
    lambda s: (
        "Preview Agent"
        if s["build_success"]
        else (
            END
            if s["retry_count"] >= 3
            else "Reviewer Agent"
        )
    ),
    {
        "Preview Agent": "Preview Agent",
        "Reviewer Agent": "Reviewer Agent",
        END: END,
    },
)

builder.add_edge("Reviewer Agent", "Fix Agent")
builder.add_edge("Fix Agent", "Build Agent")
builder.add_edge("Preview Agent", END)

modify_doc_graph = builder.compile()

from IPython.display import Image, display

display(
    Image(
        modify_doc_graph.get_graph().draw_mermaid_png()
    )
)